# WaferGuard — Wafer Anomaly Detector Audit

**Purpose:** audit and repair the supplied supervised autoencoder, then evaluate on a chronologically later test period at natural defect prevalence.

**Release posture:** metrics must be produced by running this notebook; a high score alone is not sufficient for release.

## 1. Environment

Install only the packages used by this notebook. In Colab, run this cell once if any dependency is missing.

In [ ]:
%pip -q install "torch>=2.2" "scikit-learn>=1.2" pandas numpy matplotlib joblib

## 2. Imports and reproducibility

Set deterministic seeds and select the available device.

In [ ]:
import os, random, json, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix, precision_recall_curve,
    classification_report
)

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device={DEVICE}")

## 3. Load and verify the challenge dataset

Place `secom_telemetry_modified.csv` in the notebook's working directory or upload it to the Colab session. The evaluator-only file is not used.

In [ ]:
DATA_PATHS = [
    Path("secom_telemetry_modified.csv"),
    Path("/content/secom_telemetry_modified.csv"),
    Path("/mnt/data/secom_telemetry_modified.csv"),
]
DATA_PATH = next((p for p in DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Upload secom_telemetry_modified.csv to the notebook session "
        "or place it beside this notebook."
    )

df = pd.read_csv(DATA_PATH)
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
required = {"label", "timestamp"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")
if len(sensor_cols) != 590:
    raise ValueError(f"Expected 590 sensor columns, found {len(sensor_cols)}")
if not set(df["label"].dropna().unique()).issubset({-1, 1}):
    raise ValueError("Labels must be -1 (pass) or +1 (defect).")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
df["target"] = (df["label"] == 1).astype(np.int64)

print(f"Rows: {len(df):,} | sensor features: {len(sensor_cols)}")
print(f"Label counts: {df['label'].value_counts().sort_index().to_dict()}")
print(f"Natural defect prevalence: {df['target'].mean():.2%}")
print(f"Missing sensor cells: {int(df[sensor_cols].isna().sum().sum()):,}")
print(f"Time range: {df['timestamp'].min()} → {df['timestamp'].max()}")

## 4. Audit the PS-L legacy encoding and split by time

The challenge states that `sensor_0 == 0.0` identifies PS-L legacy wafers, and `sensor_1` then stores firmware categories (100, 200, 300), not continuous measurements. We preserve those categories as three binary features, mark the two overloaded sensor fields missing for legacy rows, and split chronologically before fitting any preprocessing.

The 60/20/20 split is deliberately temporal: validation and test wafers occur after training wafers. No SMOTE is applied; class weighting is confined to training loss so validation/test retain natural prevalence.

In [ ]:
X_sensor = df[sensor_cols].to_numpy(dtype=np.float64)
y = df["target"].to_numpy(dtype=np.int64)

legacy_mask = df["sensor_0"].eq(0.0).to_numpy()
firmware_values = df["sensor_1"].to_numpy(dtype=np.float64)
firmware_codes = (100.0, 200.0, 300.0)
firmware_features = np.column_stack([
    (legacy_mask & (firmware_values == code)).astype(np.float64)
    for code in firmware_codes
])

# The sentinel and firmware codes are not continuous sensor measurements.
X_sensor[legacy_mask, 0] = np.nan
X_sensor[legacy_mask, 1] = np.nan

n = len(df)
train_end = int(n * 0.60)
val_end = int(n * 0.80)
if min(y[:train_end].sum(), y[train_end:val_end].sum(), y[val_end:].sum()) == 0:
    raise ValueError("At least one temporal split has no defect examples; revise split design.")

X_train_raw, X_val_raw, X_test_raw = (
    X_sensor[:train_end], X_sensor[train_end:val_end], X_sensor[val_end:]
)
C_train, C_val, C_test = (
    firmware_features[:train_end],
    firmware_features[train_end:val_end],
    firmware_features[val_end:]
)
y_train, y_val, y_test = y[:train_end], y[train_end:val_end], y[val_end:]

split_summary = pd.DataFrame([
    {"split":"train", "n":len(y_train), "defects":int(y_train.sum()),
     "prevalence":float(y_train.mean()), "start":str(df.timestamp.iloc[0]),
     "end":str(df.timestamp.iloc[train_end-1])},
    {"split":"validation", "n":len(y_val), "defects":int(y_val.sum()),
     "prevalence":float(y_val.mean()), "start":str(df.timestamp.iloc[train_end]),
     "end":str(df.timestamp.iloc[val_end-1])},
    {"split":"test", "n":len(y_test), "defects":int(y_test.sum()),
     "prevalence":float(y_test.mean()), "start":str(df.timestamp.iloc[val_end]),
     "end":str(df.timestamp.iloc[-1])},
])
display(split_summary)
print("PS-L rows:", int(legacy_mask.sum()))
print("PS-L firmware counts:", {
    int(code): int((legacy_mask & (firmware_values == code)).sum())
    for code in firmware_codes
})
print("PS-L labels:", df.loc[legacy_mask, "label"].value_counts().to_dict())

## 5. Fit preprocessing on training data only

Imputation statistics and scaling parameters are learned exclusively from the training period. The same fitted transformations are applied to validation and test data. Firmware indicators are appended after scaling so their category meaning is not distorted.

In [ ]:
imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

X_train = np.concatenate([X_train_scaled, C_train], axis=1).astype(np.float32)
X_val = np.concatenate([X_val_scaled, C_val], axis=1).astype(np.float32)
X_test = np.concatenate([X_test_scaled, C_test], axis=1).astype(np.float32)

assert X_train.shape[1] == len(sensor_cols) + 3
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
print("Feature count after explicit firmware encoding:", X_train.shape[1])
print("Preprocessing fit rows:", len(X_train))
print("Validation/test rows used to fit preprocessing: 0")

## 6. Build loaders and verify tensor shapes

The original notebook added a channel axis to inputs, making them `[batch, 1, features]`, while the decoder returned `[batch, features]`. Subtraction broadcast across the batch dimension and could create an unintended `[batch, batch, features]` loss tensor. This repaired pipeline keeps inputs and reconstructions as `[batch, features]` and asserts exact shape equality.

In [ ]:
BATCH_SIZE = 64
EPOCHS = 60
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
HIDDEN = 256
LATENT = 32
PATIENCE = 10

def make_loader(X, y, shuffle=False):
    xt = torch.from_numpy(np.asarray(X, dtype=np.float32))
    yt = torch.from_numpy(np.asarray(y, dtype=np.float32))
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(
        TensorDataset(xt, yt), batch_size=BATCH_SIZE,
        shuffle=shuffle, generator=generator, drop_last=False
    )

train_loader = make_loader(X_train, y_train, shuffle=True)
train_eval_loader = make_loader(X_train, y_train)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

xb, yb = next(iter(train_loader))
print("Input batch:", tuple(xb.shape), "| target batch:", tuple(yb.shape))
assert xb.ndim == 2
assert xb.shape[0] == yb.shape[0]

## 7. Supervised autoencoder

The encoder learns a compact representation; the decoder reconstructs each wafer's own feature vector; the classification head predicts defect probability. Reconstruction loss is computed per wafer on matching shapes. Class weighting addresses imbalance in the training loss without synthesizing rows or changing evaluation prevalence.

In [ ]:
class WaferSAE(nn.Module):
    def __init__(self, d_in, d_hidden=256, d_latent=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(d_hidden, d_hidden // 2),
            nn.LayerNorm(d_hidden // 2),
            nn.GELU(),
            nn.Linear(d_hidden // 2, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, d_hidden // 2), nn.GELU(),
            nn.Linear(d_hidden // 2, d_hidden), nn.GELU(),
            nn.Linear(d_hidden, d_in),
        )
        self.classifier = nn.Linear(d_latent, 1)

    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        logits = self.classifier(z).squeeze(-1)
        return reconstruction, logits, z

model = WaferSAE(X_train.shape[1], HIDDEN, LATENT).to(DEVICE)

# Reproducible demonstration of the original broadcasting defect.
demo_b, demo_f = 8, X_train.shape[1]
old_input_shape = (demo_b, 1, demo_f)
old_reconstruction_shape = (demo_b, demo_f)
broadcast_shape = torch.broadcast_shapes(old_input_shape, old_reconstruction_shape)
print("Original input/reconstruction shapes:", old_input_shape, old_reconstruction_shape)
print("Original subtraction broadcast shape:", tuple(broadcast_shape))
print("Corrected input/reconstruction shapes:", (demo_b, demo_f), (demo_b, demo_f))
assert broadcast_shape == (demo_b, demo_b, demo_f)

with torch.no_grad():
    recon, logits, latent = model(xb[:8].to(DEVICE))
print("Model output shapes:", tuple(recon.shape), tuple(logits.shape), tuple(latent.shape))
assert recon.shape == xb[:8].shape
assert logits.shape == yb[:8].shape

## 8. Train with validation-based early stopping

Only training data updates model weights. Validation average precision selects the checkpoint; validation labels also determine the operating threshold later. The test set remains untouched until final evaluation.

In [ ]:
positive = float(y_train.sum())
negative = float(len(y_train) - y_train.sum())
pos_weight = torch.tensor([negative / max(positive, 1.0)], dtype=torch.float32, device=DEVICE)
criterion_clf = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "val_ap": []}
best_ap = -np.inf
best_state = None
best_epoch = 0
epochs_without_improvement = 0

def predict_probabilities(loader):
    model.eval()
    out = []
    with torch.no_grad():
        for features, _ in loader:
            _, logits, _ = model(features.to(DEVICE))
            out.extend(torch.sigmoid(logits).cpu().numpy().tolist())
    return np.asarray(out, dtype=float)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    seen = 0
    for features, labels in train_loader:
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        reconstruction, logits, _ = model(features)
        assert reconstruction.shape == features.shape
        assert logits.shape == labels.shape

        # Per-element reconstruction loss, reduced to one scalar.
        recon_loss = torch.mean((reconstruction - features) ** 2)
        clf_loss = criterion_clf(logits, labels)
        loss = recon_loss + clf_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        grad_values = [p.grad for p in model.parameters() if p.grad is not None]
        assert all(torch.isfinite(g).all() for g in grad_values), "Non-finite gradients"
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running += loss.item() * len(labels)
        seen += len(labels)

    scheduler.step()
    val_probs = predict_probabilities(val_loader)
    val_ap = average_precision_score(y_val, val_probs)
    history["train_loss"].append(running / seen)
    history["val_ap"].append(float(val_ap))

    if val_ap > best_ap + 1e-6:
        best_ap = float(val_ap)
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | train loss={running/seen:.4f} | val AP={val_ap:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")
model.load_state_dict(best_state)
print(f"Selected epoch: {best_epoch} | best validation AP: {best_ap:.4f}")

## 9. Select a decision threshold on validation data

The threshold is selected by maximizing validation F1 over the precision-recall curve. This uses validation predictions only; the final test labels are not involved in threshold selection. Because the validation period contains relatively few defects, the selected threshold may be unstable and is documented as a residual risk.

In [ ]:
val_probs = predict_probabilities(val_loader)
precision_curve, recall_curve, thresholds = precision_recall_curve(y_val, val_probs)

if len(thresholds) == 0:
    threshold = 0.5
else:
    f1_values = (
        2 * precision_curve[:-1] * recall_curve[:-1]
        / np.maximum(precision_curve[:-1] + recall_curve[:-1], 1e-12)
    )
    best_idx = int(np.nanargmax(f1_values))
    threshold = float(thresholds[best_idx])

val_pred = (val_probs >= threshold).astype(int)
print(f"Validation-selected threshold: {threshold:.6f}")
print(f"Validation F1 at selected threshold: {f1_score(y_val, val_pred, zero_division=0):.4f}")
print(f"Validation precision: {precision_score(y_val, val_pred, zero_division=0):.4f}")
print(f"Validation recall: {recall_score(y_val, val_pred, zero_division=0):.4f}")

## 10. Final test evaluation at natural prevalence

Evaluate the selected checkpoint and validation-selected threshold once on the chronologically later test set. No balancing or threshold tuning is performed on the test set.

In [ ]:
test_probs = predict_probabilities(test_loader)
test_pred = (test_probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()
metrics = {
    "test_rows": int(len(y_test)),
    "test_defects": int(y_test.sum()),
    "test_prevalence": float(y_test.mean()),
    "threshold": float(threshold),
    "precision": float(precision_score(y_test, test_pred, zero_division=0)),
    "recall": float(recall_score(y_test, test_pred, zero_division=0)),
    "f1": float(f1_score(y_test, test_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, test_probs)) if len(np.unique(y_test)) == 2 else None,
    "average_precision": float(average_precision_score(y_test, test_probs)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
}
print(json.dumps(metrics, indent=2))
print("\nClassification report:")
print(classification_report(y_test, test_pred, target_names=["pass", "defect"], zero_division=0))

plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["val_ap"], label="validation average precision")
plt.xlabel("Epoch")
plt.ylabel("Average precision")
plt.title("Validation average precision")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Audit findings, artifacts, and release recommendation

The audit report is generated from the measured run. It distinguishes verified implementation defects from residual evaluation limitations. A release is not approved solely because a numeric F1 threshold is crossed.

In [ ]:
audit_text = f'''# WaferGuard Audit

## Scope and measured evaluation

Chronological split: 60% training / 20% validation / 20% test.
The test set is not resampled and retains its observed prevalence.

- Test rows: {metrics["test_rows"]}
- Test defects: {metrics["test_defects"]}
- Test defect prevalence: {metrics["test_prevalence"]:.2%}
- Validation-selected threshold: {metrics["threshold"]:.6f}
- Precision: {metrics["precision"]:.4f}
- Recall: {metrics["recall"]:.4f}
- F1: {metrics["f1"]:.4f}
- ROC-AUC: {"not defined" if metrics["roc_auc"] is None else f'{metrics["roc_auc"]:.4f}'}
- Average precision: {metrics["average_precision"]:.4f}
- Confusion matrix counts: TN={tn}, FP={fp}, FN={fn}, TP={tp}

## Findings ranked by severity

1. **Critical — evaluation leakage in the original notebook.** The original code fitted mean imputation and standardisation on all rows, ran SMOTE on all rows, and only then created a random train/test split. Consequently, synthetic or related information from would-be test rows could influence training, and the test distribution was artificially balanced. The repaired pipeline splits chronologically first, fits preprocessing on training rows only, does not use SMOTE, and evaluates on original test wafers at natural prevalence.

2. **Critical — invalid reconstruction-loss broadcasting.** The original loader created inputs with shape `[batch, 1, features]`, while the decoder returned `[batch, features]`. PyTorch broadcasting makes their subtraction shape `[batch, batch, features]`, coupling each reconstruction to every sample in the batch. The repaired loader uses `[batch, features]`; explicit assertions verify reconstruction and input shapes match. The reproducible shape check in this notebook demonstrates the original broadcast shape.

3. **High — invariant blindness to the PS-L tool regime.** The original pipeline treated the legacy sentinel and firmware codes as ordinary continuous sensor values, without representing the station's categorical firmware state. This can make the learned representation impose an artificial numeric ordering on categories and obscure tool-regime-specific behaviour. The repaired preprocessing marks the two overloaded sensor values missing for legacy rows and appends three binary firmware indicators. The notebook reports counts for each category so this treatment is auditable.

4. **High — threshold and class balance did not reflect deployment evaluation.** The original fixed 0.5 threshold was evaluated on a SMOTE-balanced test set. The repaired threshold is selected using validation data only, and test metrics are calculated on the untouched chronological test period at its observed defect prevalence.

## Residual risks

- The dataset is small and contains few defects in the later validation/test periods; point estimates and the validation-selected threshold may be unstable.
- Chronological holdout is more consistent with forward deployment than a random split, but one historical period does not establish robustness to all future drift.
- The PS-L representation is a task-brief-driven repair for invariant blindness to the categorical tool regime; it should be validated against station metadata and process-integration expectations before production use.
- This is a challenge-level audit, not a substitute for prospective monitoring, calibration, drift alerts, and production acceptance testing.

## Release recommendation

**NO-GO for production pending further validation.** Use the measured test metrics above as the honest challenge evaluation. The corrected pipeline removes identified leakage and tensor-broadcasting defects, but the limited number of defects and single temporal holdout are insufficient to certify production readiness.
'''
Path("AUDIT.md").write_text(audit_text, encoding="utf-8")

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_guard_fixed.pt")
import joblib
joblib.dump(
    {"imputer": imputer, "scaler": scaler, "sensor_columns": sensor_cols,
     "firmware_codes": firmware_codes, "threshold": threshold},
    "artifacts/preprocessor.joblib"
)
Path("artifacts/test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Wrote AUDIT.md, artifacts/wafer_guard_fixed.pt, artifacts/preprocessor.joblib, and artifacts/test_metrics.json")
print("\nAUDIT.md preview:\n")
print(audit_text)